## 1. Setup & Data Loading

In [1]:
import os
os.environ.pop('MPLBACKEND', None)   # FIX backend issue
os.environ['MPLBACKEND'] = 'tkagg'   # or 'agg' if needed

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats, signal
from scipy.fft import fft, fftfreq
from itertools import combinations
from collections import Counter, defaultdict

# Change point detection
import ruptures as rpt

# Sequence mining
try:
    from prefixspan import PrefixSpan
    PREFIXSPAN_AVAILABLE = True
    print('✅ PrefixSpan available')
except ImportError:
    PREFIXSPAN_AVAILABLE = False
    print('⚠️ PrefixSpan not found — custom implementation will be used')

# Association rules
try:
    from mlxtend.frequent_patterns import fpgrowth, association_rules
    from mlxtend.preprocessing import TransactionEncoder
    MLXTEND_AVAILABLE = True
    print('✅ mlxtend available')
except ImportError:
    MLXTEND_AVAILABLE = False
    print('⚠️ mlxtend not available — manual Apriori used')

SEED = 42
np.random.seed(SEED)

plt.rcParams.update({
    'figure.figsize': (14, 5),
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'axes.grid': True,
    'grid.alpha': 0.3
})

print('✅ Setup complete')

⚠️ PrefixSpan not found — custom implementation will be used
⚠️ mlxtend not available — manual Apriori used
✅ Setup complete


In [2]:
# ── Load data ─────────────────────────────────────────────────────────────
try:
    df = pd.read_csv('temporal_features.csv', low_memory=False)
    df['time'] = pd.to_datetime(df['time'], utc=True, errors='coerce')
    df = df.sort_values('time').reset_index(drop=True)
    print(f'✅ Loaded temporal_features.csv: {df.shape}')
except FileNotFoundError:
    print('⚠️  temporal_features.csv not found — loading raw dataset')
    df = pd.read_csv('final_preprocessed_earthquake_data.csv', low_memory=False)
    df['time'] = pd.to_datetime(df['time'], utc=True, errors='coerce')
    if 'type' in df.columns:
        df = df[df['type']=='earthquake']
    df['inter_event_time_hrs'] = df['time'].diff().dt.total_seconds().fillna(0) / 3600
    df['log_iet'] = np.log1p(df['inter_event_time_hrs'])
    df['is_aftershock'] = 0
    df = df.sort_values('time').reset_index(drop=True)

# Ensure grid_id exists
if 'grid_id' not in df.columns:
    df['lat_bin'] = (df['latitude'] // 2).astype(int)
    df['lon_bin'] = (df['longitude'] // 2).astype(int)
    df['grid_id'] = df['lat_bin'].astype(str) + '_' + df['lon_bin'].astype(str)

print(f'Columns: {df.columns.tolist()}')
print(f'Date range: {df["time"].min().date()} → {df["time"].max().date()}')

✅ Loaded temporal_features.csv: (3153, 32)
Columns: ['time', 'latitude', 'longitude', 'depth', 'mag', 'year', 'month', 'day_of_year', 'hour', 'day_of_week', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'doy_sin', 'doy_cos', 'inter_event_time_hrs', 'log_iet', 'iet_per_cell_hrs', 'time_since_major_hrs', 'is_aftershock', 'rolling_count_7d', 'rolling_count_30d', 'rolling_count_90d', 'rolling_mag_7d', 'rolling_mag_30d', 'rolling_mag_90d', 'rolling_max_mag_30d', 'rolling_std_mag_30d', 'b_value_rolling', 'omori_decay_rate', 'grid_id']
Date range: 2000-01-01 → 2025-05-26


## 2. Sequence Mining — PrefixSpan on Discretised Magnitude Sequences

In [3]:
# ── Discretise magnitude into 3 symbols ───────────────────────────────────
# L = Low (M < 5), M = Moderate (5 ≤ M < 6), H = High (M ≥ 6)
def discretise_mag(m):
    if m < 5.0: return 'L'
    elif m < 6.0: return 'M'
    else: return 'H'

df['mag_sym'] = df['mag'].apply(discretise_mag)

print('Symbol distribution:')
vc = df['mag_sym'].value_counts()
for sym, cnt in vc.items():
    print(f'  {sym}: {cnt:>8,} ({cnt/len(df)*100:.1f}%)')

Symbol distribution:
  L:    2,348 (74.5%)
  M:      759 (24.1%)
  H:       46 (1.5%)


In [4]:
# FIX: ensure 'time' column exists and is datetime

# check columns
print(df.columns)

# if time column has different name → fix it
# (uncomment the correct one if needed)
# df.rename(columns={'datetime': 'time'}, inplace=True)
# df.rename(columns={'date': 'time'}, inplace=True)

# convert to datetime (IMPORTANT)
df['time'] = pd.to_datetime(df['time'], errors='coerce')

# drop invalid rows
df = df.dropna(subset=['time'])

# ── Build sequences per grid cell (monthly windows) ────────────────────────

sequences = []
seq_meta  = []

df['yr_month'] = df['time'].dt.to_period('M')

grp = df.groupby(['grid_id', 'yr_month'])
for (gid, ym), gdf in grp:
    seq = gdf.sort_values('time')['mag_sym'].tolist()
    if len(seq) >= 3:
        sequences.append(seq)
        seq_meta.append({'grid_id': gid, 'month': str(ym)})

print(f'Total sequences: {len(sequences):,}')
print(f'Avg sequence length: {np.mean([len(s) for s in sequences]):.1f}')
print('Example sequences:')
for i, s in enumerate(sequences[:5]):
    print(f'{i+1}: {s[:10]}...' if len(s)>10 else f'{i+1}: {s}')

Index(['time', 'latitude', 'longitude', 'depth', 'mag', 'year', 'month',
       'day_of_year', 'hour', 'day_of_week', 'hour_sin', 'hour_cos',
       'month_sin', 'month_cos', 'doy_sin', 'doy_cos', 'inter_event_time_hrs',
       'log_iet', 'iet_per_cell_hrs', 'time_since_major_hrs', 'is_aftershock',
       'rolling_count_7d', 'rolling_count_30d', 'rolling_count_90d',
       'rolling_mag_7d', 'rolling_mag_30d', 'rolling_mag_90d',
       'rolling_max_mag_30d', 'rolling_std_mag_30d', 'b_value_rolling',
       'omori_decay_rate', 'grid_id', 'mag_sym'],
      dtype='object')
Total sequences: 176
Avg sequence length: 5.6
Example sequences:
1: ['M', 'M', 'L']
2: ['L', 'L', 'L']
3: ['M', 'L', 'L', 'M', 'M', 'M']
4: ['L', 'L', 'L']
5: ['L', 'L', 'L']


In [5]:
# ── PrefixSpan Mining ─────────────────────────────────────────────────────

if PREFIXSPAN_AVAILABLE:
    # Convert symbols to integers for PrefixSpan
    sym_map = {'L': 0, 'M': 1, 'H': 2}
    int_seqs = [[sym_map[s] for s in seq] for seq in sequences]

    MIN_SUPPORT = max(2, int(0.02 * len(sequences)))  # 2% support
    print(f'Running PrefixSpan with min_support={MIN_SUPPORT} '
          f'({MIN_SUPPORT/len(sequences)*100:.1f}% of {len(sequences)} sequences)')

    ps = PrefixSpan(int_seqs)
    ps.minlen = 2
    ps.maxlen = 5
    patterns_raw = ps.frequent(MIN_SUPPORT)

    # Convert back to symbols
    rev_map = {0: 'L', 1: 'M', 2: 'H'}
    patterns = [(sup, [rev_map[x] for x in pat]) for sup, pat in patterns_raw]
    patterns_sorted = sorted(patterns, key=lambda x: -x[0])

    print(f'\n✅ Found {len(patterns_sorted)} frequent sequences')
    print(f'\nTop 20 frequent sequences:')
    print(f'{"Rank":<5} {"Support":<10} {"Pattern"}')
    print('-' * 40)
    for rank, (sup, pat) in enumerate(patterns_sorted[:20], 1):
        support_pct = sup / len(sequences) * 100
        print(f'{rank:<5} {sup:<6} ({support_pct:4.1f}%)  {" → ".join(pat)}')

else:
    # ── Custom PrefixSpan implementation ──────────────────────────────────
    print('Using custom sequence mining implementation')

    def count_subsequences(sequences, pattern):
        """Count how many sequences contain 'pattern' as a subsequence."""
        count = 0
        for seq in sequences:
            # greedy subsequence check
            pi = 0
            for item in seq:
                if pi < len(pattern) and item == pattern[pi]:
                    pi += 1
            if pi == len(pattern):
                count += 1
        return count

    symbols  = ['L', 'M', 'H']
    MIN_SUP  = max(2, int(0.02 * len(sequences)))
    all_pats = []

    # Length-2 and length-3 patterns
    for a in symbols:
        for b in symbols:
            pat = [a, b]
            sup = count_subsequences(sequences, pat)
            if sup >= MIN_SUP:
                all_pats.append((sup, pat))
                for c in symbols:
                    pat3 = [a, b, c]
                    sup3 = count_subsequences(sequences, pat3)
                    if sup3 >= MIN_SUP:
                        all_pats.append((sup3, pat3))

    patterns_sorted = sorted(all_pats, key=lambda x: -x[0])
    print(f'\n✅ Found {len(patterns_sorted)} frequent sequences')
    print(f'\nTop 15 frequent patterns:')
    print(f'{"Rank":<5} {"Support":<10} {"Pattern"}')
    print('-' * 40)
    for rank, (sup, pat) in enumerate(patterns_sorted[:15], 1):
        print(f'{rank:<5} {sup:<8} {" → ".join(pat)}')

Using custom sequence mining implementation

✅ Found 28 frequent sequences

Top 15 frequent patterns:
Rank  Support    Pattern
----------------------------------------
1     165      L → L
2     133      L → L → L
3     95       M → L
4     88       L → M
5     73       M → L → L
6     70       L → M → L
7     64       L → L → M
8     61       M → M
9     49       M → M → L
10    44       M → L → M
11    42       L → M → M
12    28       M → M → M
13    11       H → M
14    10       H → L
15    7        L → H


In [6]:
# ═══════════════════════════════════════════════════════════════
# PLOT MINE-3: Magnitude Sequence Transition Network
# ═══════════════════════════════════════════════════════════════
import seaborn as sns, matplotlib.pyplot as plt, numpy as np

syms = ['L','M','H']
tr = {s:{t:0 for t in syms} for s in syms}
for seq in sequences:
    for a, b in zip(seq, seq[1:]):
        if a in tr and b in tr[a]: tr[a][b] += 1

tp = np.array([[tr[s][t]/(sum(tr[s].values()) or 1) for t in syms] for s in syms])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Magnitude Sequence Transition Analysis', fontsize=13, fontweight='bold')

sns.heatmap(tp, annot=True, fmt='.3f', cmap='YlOrRd',
            xticklabels=['Low','Moderate','High'], yticklabels=['Low','Moderate','High'],
            ax=axes[0], linewidths=0.5, annot_kws={'size':13})
axes[0].set_title('Bigram Transition Probability Matrix', fontweight='bold')
axes[0].set_xlabel('Next State'); axes[0].set_ylabel('Current State')

ax2 = axes[1]
sl = [len(s) for s in sequences]
ax2.hist(sl, bins=50, color='steelblue', edgecolor='white', alpha=0.85)
ax2.axvline(np.mean(sl), color='red', linestyle='--', lw=2, label=f'Mean={np.mean(sl):.1f}')
ax2.set_title('Magnitude Sequence Length Distribution', fontweight='bold')
ax2.set_xlabel('Sequence Length'); ax2.set_ylabel('Count')
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('plot_MINE3_sequence_transitions.png', dpi=120, bbox_inches='tight')
plt.show()
print('Plot MINE-3: Sequence Transitions saved')

Plot MINE-3: Sequence Transitions saved


In [7]:
# ── Sequence pattern visualisation ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Top-15 patterns — horizontal bar chart
top_n = min(15, len(patterns_sorted))
top_pats = patterns_sorted[:top_n]
labels  = [' → '.join(p) for _, p in top_pats]
supports = [s for s, _ in top_pats]
support_pcts = np.array(supports) / len(sequences) * 100

color_map = {'L → L': '#2ecc71', 'L → M': '#f39c12', 'L → H': '#e74c3c',
             'M': '#f39c12', 'H': '#e74c3c'}
bar_colors = ['#e74c3c' if 'H' in l else '#f39c12' if 'M' in l else '#2ecc71' for l in labels]

bars = axes[0].barh(labels[::-1], support_pcts[::-1], color=bar_colors[::-1],
                    height=0.6, edgecolor='white')
for bar, pct in zip(bars, support_pcts[::-1]):
    axes[0].text(pct + 0.1, bar.get_y() + bar.get_height()/2,
                 f'{pct:.1f}%', va='center', fontsize=9)
axes[0].set_title(f'Top {top_n} Frequent Magnitude Sequences (PrefixSpan)')
axes[0].set_xlabel('Support (%)')
axes[0].set_ylabel('Sequence Pattern (→ = followed by)')

# Patch legend
legend_elems = [
    mpatches.Patch(color='#2ecc71', label='Contains only Low (L)'),
    mpatches.Patch(color='#f39c12', label='Contains Moderate (M)'),
    mpatches.Patch(color='#e74c3c', label='Contains High (H)'),
]
axes[0].legend(handles=legend_elems, loc='lower right', fontsize=9)

# Transition matrix heatmap
# Count bigram transitions
trans = Counter()
for seq in sequences:
    for a, b in zip(seq[:-1], seq[1:]):
        trans[(a, b)] += 1

syms = ['L', 'M', 'H']
T_mat = np.zeros((3, 3))
for i, a in enumerate(syms):
    for j, b in enumerate(syms):
        T_mat[i, j] = trans.get((a, b), 0)
# Normalise rows
row_sums = T_mat.sum(axis=1, keepdims=True)
T_norm = np.divide(T_mat, row_sums, where=row_sums>0)

full_names = ['Low (M<5)', 'Moderate (5-6)', 'High (M≥6)']
sns.heatmap(T_norm, annot=True, fmt='.3f', cmap='YlOrRd', ax=axes[1],
            xticklabels=full_names, yticklabels=full_names,
            linewidths=0.5, vmin=0, vmax=1)
axes[1].set_title('Magnitude Transition Probability Matrix')
axes[1].set_xlabel('Next Event Class')
axes[1].set_ylabel('Current Event Class')

plt.suptitle('Sequence Mining — Magnitude Category Patterns', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('sequence_mining.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Sequence mining visualisation saved')

✅ Sequence mining visualisation saved


## 3. Periodicity / Seasonality Mining

In [8]:
# ── Build daily event count time series ────────────────────────────────────
daily = (df.set_index('time')
          .resample('D')['mag']
          .agg(['count', 'mean', 'max'])
          .fillna(0))
daily.columns = ['daily_count', 'daily_mean_mag', 'daily_max_mag']

print(f'Daily time series: {len(daily)} days ({daily.index.min().date()} → {daily.index.max().date()})')
print(daily.describe())

Daily time series: 9278 days (2000-01-01 → 2025-05-26)
       daily_count  daily_mean_mag  daily_max_mag
count  9278.000000     9278.000000    9278.000000
mean      0.339836        1.183527       1.196021
std       0.733372        2.076801       2.100695
min       0.000000        0.000000       0.000000
25%       0.000000        0.000000       0.000000
50%       0.000000        0.000000       0.000000
75%       0.000000        0.000000       0.000000
max      15.000000        7.300000       7.300000


In [ ]:
# ════════════════════════════════════════════════════════════════════
# PLOT MINE-ACF: ACF & PACF — Example  (Autocorrelation Analysis)
#
#  ACF  = how correlated the series is with its own lagged values
#  PACF = the 'pure' correlation at each lag (removing shorter-lag effects)
# ════════════════════════════════════════════════════════════════════
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

# Build daily event-count series (reuse 'daily' if available, else rebuild)
try:
    _ts_acf = daily['daily_count'].fillna(0)
except Exception:
    _ts_acf = (df.set_index('time').resample('D')['mag']
               .count().fillna(0))

NLAGS = 60   # maximum lag to display

fig = plt.figure(figsize=(20, 10))
fig.patch.set_facecolor('white')                                       
gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.3)

# ── Panel 1: raw time series ──
ax_ts = fig.add_subplot(gs[0, :])
ax_ts.set_facecolor('white')                                           
ax_ts.plot(_ts_acf.index, _ts_acf.values,
           color='#2176ae', linewidth=0.9, alpha=0.85)                 # ← darkened blue
_roll = _ts_acf.rolling(30, min_periods=1).mean()
ax_ts.plot(_ts_acf.index, _roll.values,
           color='#d4820a', linewidth=2, label='30-day Rolling Mean')  # ← darkened orange
ax_ts.set_title('Daily Earthquake Count (input series)',
                color='black', fontsize=12, fontweight='bold')         
ax_ts.set_ylabel('Count', color='#333', fontsize=10)                  
ax_ts.tick_params(colors='#333')                                       
ax_ts.legend(facecolor='white', edgecolor='#bbb',                     
             labelcolor='black', fontsize=9)                           
for sp in ax_ts.spines.values(): sp.set_edgecolor('#bbb')             

# ── Panel 2: ACF ──
ax_acf = fig.add_subplot(gs[1, 0])
ax_acf.set_facecolor('white')                                          
plot_acf(_ts_acf.values, lags=NLAGS, ax=ax_acf,
         color='#2176ae', vlines_kwargs={'colors': '#2176ae'},         # ← darkened blue
         alpha=0.05, zero=False)
ax_acf.set_title('ACF — Autocorrelation Function',
                 color='black', fontsize=11, fontweight='bold')        
ax_acf.set_xlabel('Lag (days)', color='#333', fontsize=9)             
ax_acf.set_ylabel('Autocorrelation', color='#333', fontsize=9)        
ax_acf.tick_params(colors='#333')                                      
ax_acf.axhline(0, color='#aaa', linewidth=0.8)                        
for sp in ax_acf.spines.values(): sp.set_edgecolor('#bbb')            

# Annotate significant lags (weekly ~7, monthly ~30)
for lag_mark, lbl in [(7, '7d'), (14, '14d'), (30, '30d')]:
    if lag_mark <= NLAGS:
        ax_acf.axvline(lag_mark, color='#c0392b', linewidth=1,        # ← darkened red
                       linestyle='--', alpha=0.7)
        ax_acf.text(lag_mark + 0.5, ax_acf.get_ylim()[1] * 0.9,
                    lbl, color='#c0392b', fontsize=8)                  # ← darkened red

# ── Panel 3: PACF ──
ax_pacf = fig.add_subplot(gs[1, 1])
ax_pacf.set_facecolor('white')                                         
plot_pacf(_ts_acf.values, lags=NLAGS, ax=ax_pacf,
          color='#d4820a', vlines_kwargs={'colors': '#d4820a'},        # ← darkened orange
          alpha=0.05, zero=False, method='ywm')
ax_pacf.set_title('PACF — Partial Autocorrelation Function',
                  color='black', fontsize=11, fontweight='bold')       
ax_pacf.set_xlabel('Lag (days)', color='#333', fontsize=9)            
ax_pacf.set_ylabel('Partial Autocorrelation', color='#333', fontsize=9) 
ax_pacf.tick_params(colors='#333')                                     
ax_pacf.axhline(0, color='#aaa', linewidth=0.8)                       
for sp in ax_pacf.spines.values(): sp.set_edgecolor('#bbb')           

for lag_mark, lbl in [(7, '7d'), (14, '14d'), (30, '30d')]:
    if lag_mark <= NLAGS:
        ax_pacf.axvline(lag_mark, color='#6d28d9', linewidth=1,       # ← darkened purple
                        linestyle='--', alpha=0.7)
        ax_pacf.text(lag_mark + 0.5, ax_pacf.get_ylim()[1] * 0.9,
                     lbl, color='#6d28d9', fontsize=8)                # ← darkened purple

fig.suptitle('📉  ACF & PACF Plots: Example — Daily Seismic Event Count\n'
             'ACF reveals overall periodicity; PACF reveals direct lag dependencies',
             color='black', fontsize=13, fontweight='bold', y=1.01)    

plt.savefig('plot_MINE_acf_pacf.png', dpi=150,
            bbox_inches='tight', facecolor='white')                    
plt.show()
print('✅  ACF/PACF example plot saved.')


✅  ACF/PACF example plot saved.


In [10]:
# ── FFT Periodicity Analysis ───────────────────────────────────────────────
signal_counts = daily['daily_count'].values - daily['daily_count'].mean()

N   = len(signal_counts)
dt  = 1  # 1 day sampling interval
fft_vals  = np.abs(fft(signal_counts)[:N//2])   # one-sided amplitude
freqs     = fftfreq(N, d=dt)[:N//2]             # cycles/day

# Convert frequency to period in days
with np.errstate(divide='ignore', invalid='ignore'):
    periods = np.where(freqs > 0, 1.0 / freqs, np.inf)

# Top peaks (excluding DC component and very long periods)
valid_mask = (periods > 2) & (periods < 3650)
fft_valid  = fft_vals[valid_mask]
per_valid  = periods[valid_mask]

top_idx = np.argsort(fft_valid)[-20:][::-1]
top_periods  = per_valid[top_idx]
top_amps     = fft_valid[top_idx]

print('Top 10 dominant periodicities:')
print(f'{"Rank":<5} {"Period (days)":<18} {"Amplitude"}')
print('-' * 40)
for rank, (per, amp) in enumerate(zip(top_periods[:10], top_amps[:10]), 1):
    months = per / 30.44
    print(f'{rank:<5} {per:>10.1f} days  ({months:.1f} months)   {amp:.1f}')

Top 10 dominant periodicities:
Rank  Period (days)      Amplitude
----------------------------------------
1         3092.7 days  (101.6 months)   330.0
2         1546.3 days  (50.8 months)   274.6
3          386.6 days  (12.7 months)   222.3
4            2.9 days  (0.1 months)   202.5
5         1325.4 days  (43.5 months)   201.3
6         2319.5 days  (76.2 months)   194.7
7            2.3 days  (0.1 months)   182.0
8           14.8 days  (0.5 months)   167.7
9           14.9 days  (0.5 months)   166.8
10          14.9 days  (0.5 months)   165.3


In [13]:
# ── Seasonal decomposition ─────────────────────────────────────────────────
from statsmodels.tsa.seasonal import seasonal_decompose

# STL needs at least 2 full cycles
decomp = seasonal_decompose(
    daily['daily_count'],
    model='additive',
    period=365,     # annual periodicity
    extrapolate_trend='freq'
)

In [14]:
# ═══════════════════════════════════════════════════════════════
# PLOT MINE-2: Monthly Seasonality Heatmap
# ═══════════════════════════════════════════════════════════════
import seaborn as sns, matplotlib.pyplot as plt, numpy as np

df_s = df.copy()
df_s['yr'] = df_s['time'].dt.year
df_s['mo'] = df_s['time'].dt.month
piv = df_s.groupby(['yr','mo']).size().unstack(fill_value=0)
mo_labs = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
piv.columns = [mo_labs[m-1] for m in piv.columns]

fig, axes = plt.subplots(1, 2, figsize=(20, 8))
fig.suptitle('Temporal Seasonality Analysis', fontsize=14, fontweight='bold')

sns.heatmap(piv, cmap='YlOrRd', ax=axes[0], linewidths=0.1, linecolor='white',
            cbar_kws={'label':'Monthly Event Count'})
axes[0].set_title('Year × Month Event Count Heatmap', fontweight='bold')
axes[0].set_xlabel('Month'); axes[0].set_ylabel('Year')

ax2 = axes[1]
avg = piv.mean(); std = piv.std()
x = np.arange(12)
ax2.bar(x, avg, color=plt.cm.coolwarm(np.linspace(0.1,0.9,12)), edgecolor='white', alpha=0.85, label='Mean')
ax2.errorbar(x, avg, yerr=std, fmt='none', color='black', capsize=5, lw=1.5, label='±1 Std')
ax2.set_xticks(x); ax2.set_xticklabels(mo_labs, rotation=45)
ax2.set_title('Average Monthly Event Count (All Years)', fontweight='bold')
ax2.set_xlabel('Month'); ax2.set_ylabel('Avg Event Count')
ax2.legend(); ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('plot_MINE2_seasonality_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()
print('Plot MINE-2: Seasonality Heatmap saved')

Plot MINE-2: Seasonality Heatmap saved


In [15]:
# ── Periodicity visualisation ─────────────────────────────────────────────
fig = plt.figure(figsize=(20, 12))
gs  = gridspec.GridSpec(3, 2, figure=fig, hspace=0.45, wspace=0.3)

# 1. FFT periodogram
ax1 = fig.add_subplot(gs[0, :])
ax1.semilogx(per_valid, fft_valid, color='steelblue', linewidth=0.8, alpha=0.8)
# Mark top peaks
for per, amp in zip(top_periods[:5], top_amps[:5]):
    ax1.axvline(per, color='red', linestyle='--', alpha=0.6, linewidth=1)
    ax1.text(per * 1.05, amp * 0.9, f'{per:.0f}d', fontsize=9, color='red')
ax1.set_title('FFT Periodogram — Daily Earthquake Count (log-scale x-axis)')
ax1.set_xlabel('Period (days, log scale)')
ax1.set_ylabel('Amplitude')
ax1.set_xlim([2, 3600])

# 2. Raw time series
ax2 = fig.add_subplot(gs[1, 0])
ax2.plot(daily.index, decomp.observed, color='steelblue', linewidth=0.6, alpha=0.8)
ax2.set_title('Observed Daily Event Count')
ax2.set_ylabel('Count')

# 3. Trend
ax3 = fig.add_subplot(gs[1, 1])
ax3.plot(daily.index, decomp.trend, color='darkred', linewidth=1.5)
ax3.set_title('Trend Component')
ax3.set_ylabel('Count')

# 4. Seasonal
ax4 = fig.add_subplot(gs[2, 0])
# Plot single year of seasonal pattern
seas_yr = decomp.seasonal[:365]
ax4.plot(range(365), seas_yr.values, color='seagreen', linewidth=1.5)
ax4.axhline(0, color='black', linewidth=0.5)
ax4.set_title('Annual Seasonal Pattern')
ax4.set_xlabel('Day of Year')
ax4.set_ylabel('Seasonal Component')

# 5. Monthly boxplot
ax5 = fig.add_subplot(gs[2, 1])
daily['month'] = daily.index.month
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
month_data = [daily[daily['month'] == m]['daily_count'].values for m in range(1, 13)]
bp = ax5.boxplot(month_data, labels=month_names, patch_artist=True,
                 medianprops={'color': 'black', 'linewidth': 2})
colors_month = plt.cm.coolwarm(np.linspace(0, 1, 12))
for patch, color in zip(bp['boxes'], colors_month):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax5.set_title('Monthly Earthquake Count Distribution')
ax5.set_xlabel('Month')
ax5.set_ylabel('Events/day')

plt.suptitle('Periodicity & Seasonality Mining', fontsize=18, fontweight='bold', y=1.01)
plt.savefig('periodicity_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Periodicity plots saved')

NameError: name 'per_valid' is not defined

## 4. Change Point Detection — PELT

In [16]:
# ── Prepare signals for change point detection ─────────────────────────────
# Use monthly-aggregated count and mean magnitude
monthly = (df.set_index('time')
             .resample('ME')[['mag','depth']]
             .agg(count=('mag','count'), mean_mag=('mag','mean'), max_mag=('mag','max'))
             .fillna(0))

print(f'Monthly series: {len(monthly)} months')
monthly.head()

Monthly series: 305 months


,count,mean_mag,max_mag
time,,,
2000-01-31 00:00:00+00:00,8,5.050000,6.3
2000-02-29 00:00:00+00:00,2,4.900000,5.1
2000-03-31 00:00:00+00:00,1,4.600000,4.6
2000-04-30 00:00:00+00:00,2,4.850000,5.2
2000-05-31 00:00:00+00:00,7,4.842857,5.5


In [17]:
# ── PELT change point detection — event count ──────────────────────────────
count_signal = monthly['count'].values.astype(float).reshape(-1, 1)
mag_signal   = monthly['mean_mag'].values.astype(float).reshape(-1, 1)

# PELT with RBF cost
algo_count = rpt.Pelt(model='rbf', min_size=3, jump=1).fit(count_signal)
bkps_count = algo_count.predict(pen=5)

algo_mag   = rpt.Pelt(model='rbf', min_size=3, jump=1).fit(mag_signal)
bkps_mag   = algo_mag.predict(pen=3)

print(f'Change points (event count): {len(bkps_count)-1}')
print(f'Change points (mean mag)   : {len(bkps_mag)-1}')

# Map indices to dates
dates = monthly.index
cp_dates_count = [dates[i-1] for i in bkps_count[:-1]]
cp_dates_mag   = [dates[i-1] for i in bkps_mag[:-1]]

print('\nTop change points (event count):', [str(d.date()) for d in cp_dates_count[:10]])
print('Top change points (mean mag)    :', [str(d.date()) for d in cp_dates_mag[:10]])

Change points (event count): 4
Change points (mean mag)   : 3

Top change points (event count): ['2004-11-30', '2010-02-28', '2015-09-30', '2019-01-31']
Top change points (mean mag)    : ['2006-02-28', '2011-03-31', '2019-03-31']


In [18]:
# ── BOCPD — Bayesian Online Change Point Detection ────────────────────────
# Implemented as a simple online algorithm with Gaussian likelihood

def bocpd(data, hazard_lambda=250, mu0=0, kappa0=1, alpha0=1, beta0=1):
    """
    Bayesian Online Change Point Detection.
    Returns posterior probability P(run_length = t | data_1:t) for each t.
    """
    T = len(data)
    R  = np.zeros((T+1, T+1))  # R[t, r] = P(run_length=r at time t)
    R[0, 0] = 1.0

    mu_arr    = np.array([mu0])
    kappa_arr = np.array([kappa0])
    alpha_arr = np.array([alpha0])
    beta_arr  = np.array([beta0])

    maxes = np.zeros(T)

    for t in range(1, T+1):
        x = data[t-1]

        # Predictive probability (Student-t)
        df_t = 2 * alpha_arr
        scale_t = np.sqrt(beta_arr * (kappa_arr + 1) / (alpha_arr * kappa_arr))
        pred_probs = stats.t.pdf(x, df=df_t, loc=mu_arr, scale=scale_t)

        # Hazard function (constant rate)
        H = 1.0 / hazard_lambda

        # Growth probabilities
        R[t, 1:t+1] = R[t-1, :t] * pred_probs * (1 - H)
        # Change point probability
        R[t, 0]     = np.sum(R[t-1, :t] * pred_probs) * H

        # Normalise
        R[t] /= (R[t].sum() + 1e-300)

        # Update sufficient statistics
        kappa_new = kappa_arr + 1
        mu_new    = (kappa_arr * mu_arr + x) / kappa_new
        alpha_new = alpha_arr + 0.5
        beta_new  = beta_arr + (kappa_arr * (x - mu_arr)**2) / (2 * kappa_new)

        kappa_arr = np.concatenate([[kappa0],    kappa_new])
        mu_arr    = np.concatenate([[mu0],        mu_new])
        alpha_arr = np.concatenate([[alpha0],     alpha_new])
        beta_arr  = np.concatenate([[beta0],      beta_new])

        maxes[t-1] = R[t].argmax()

    return R, maxes

# Run on normalised monthly count
from sklearn.preprocessing import StandardScaler
count_norm = StandardScaler().fit_transform(count_signal).flatten()

R_bocpd, maxes = bocpd(count_norm, hazard_lambda=200)

# P(change point) at each t = P(run_length=0)
cp_prob = R_bocpd[1:, 0]

bocpd_threshold = 0.3
bocpd_cps = np.where(cp_prob > bocpd_threshold)[0]
bocpd_dates = [dates[i] for i in bocpd_cps if i < len(dates)]

print(f'BOCPD change points (threshold={bocpd_threshold}): {len(bocpd_dates)}')
print([str(d.date()) for d in bocpd_dates[:10]])

BOCPD change points (threshold=0.3): 0
[]


In [19]:
# ═══════════════════════════════════════════════════════════════
# PLOT MINE-4: Change Point Regime Summary
# ═══════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt, numpy as np

fig, axes = plt.subplots(2, 1, figsize=(18, 10))
fig.suptitle('Change Point Detection — Regime Segmentation', fontsize=14, fontweight='bold')

# Monthly count with regime colour bands
ax1 = axes[0]
r_starts = [monthly.index[0]] + list(cp_dates_count[:-1])
r_ends   = list(cp_dates_count[:-1]) + [monthly.index[-1]]
pal_r = plt.cm.Set3(np.linspace(0,1,max(len(r_starts),3)))
for sd, ed, col in zip(r_starts, r_ends, pal_r):
    ax1.axvspan(sd, ed, alpha=0.2, color=col)
ax1.fill_between(monthly.index, monthly['count'], alpha=0.45, color='steelblue')
ax1.plot(monthly.index, monthly['count'], color='steelblue', lw=1.2)
for d in cp_dates_count[:-1]:
    ax1.axvline(d, color='crimson', linestyle='--', lw=1.8, alpha=0.9)
ax1.set_title('Monthly Event Count — PELT Regime Segments', fontweight='bold')
ax1.set_xlabel('Date'); ax1.set_ylabel('Monthly Count'); ax1.grid(alpha=0.2)

# Monthly mean magnitude
ax2 = axes[1]
ax2.fill_between(monthly.index, monthly['mean_mag'], alpha=0.45, color='darkorange')
ax2.plot(monthly.index, monthly['mean_mag'], color='darkorange', lw=1.2)
cp_mag_plot = cp_dates_mag if 'cp_dates_mag' in dir() else cp_dates_count
for d in cp_mag_plot[:-1]:
    ax2.axvline(d, color='navy', linestyle='--', lw=1.8, alpha=0.9)
ax2.set_title('Monthly Mean Magnitude with Change Points', fontweight='bold')
ax2.set_xlabel('Date'); ax2.set_ylabel('Mean Magnitude'); ax2.grid(alpha=0.2)

plt.tight_layout()
plt.savefig('plot_MINE4_cpd_regimes.png', dpi=120, bbox_inches='tight')
plt.show()
print('Plot MINE-4: Change Point Regimes saved')

Plot MINE-4: Change Point Regimes saved


In [20]:
# ── Change point visualisation ────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(18, 14))

# 1. Monthly event count with PELT breakpoints
ax = axes[0]
ax.fill_between(monthly.index, monthly['count'], alpha=0.4, color='steelblue')
ax.plot(monthly.index, monthly['count'], color='steelblue', linewidth=1.0)
for d in cp_dates_count:
    ax.axvline(d, color='red', linestyle='--', linewidth=1.2, alpha=0.8)
ax.set_title(f'PELT Change Point Detection — Monthly Event Count ({len(bkps_count)-1} CPs)')
ax.set_ylabel('Events/month')
# Annotate CPs
for k, d in enumerate(cp_dates_count[:5]):
    ymax = monthly['count'].max()
    ax.text(d, ymax*0.92, f'CP{k+1}', fontsize=8, color='red', ha='center')

# 2. Mean magnitude with PELT breakpoints
ax = axes[1]
ax.plot(monthly.index, monthly['mean_mag'], color='darkorange', linewidth=1.2)
ax.fill_between(monthly.index, monthly['mean_mag'], alpha=0.3, color='orange')
for d in cp_dates_mag:
    ax.axvline(d, color='darkred', linestyle='--', linewidth=1.2, alpha=0.8)
ax.set_title(f'PELT Change Point Detection — Monthly Mean Magnitude ({len(bkps_mag)-1} CPs)')
ax.set_ylabel('Mean Magnitude')

# 3. BOCPD posterior
ax = axes[2]
t_axis = dates[:len(cp_prob)]
ax.plot(t_axis, cp_prob, color='purple', linewidth=1.0, label='P(change point)')
ax.axhline(bocpd_threshold, color='red', linestyle='--', linewidth=1.0, label=f'Threshold={bocpd_threshold}')
ax.fill_between(t_axis, cp_prob, alpha=0.3, color='purple')
ax.set_title('BOCPD — Posterior Probability of Change Point')
ax.set_ylabel('P(run_length = 0)')
ax.set_xlabel('Date')
ax.legend()

plt.suptitle('Change Point Detection — PELT & BOCPD', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('change_point_detection.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Change point plots saved')

✅ Change point plots saved


## 5. Temporal Association Rules

In [21]:
# ── Window-based temporal association mining ───────────────────────────────
# Question: does a M≥5.5 in region A within 7 days predict M≥5.5 in region B?

# Use top-50 most active grid cells for efficiency
top_cells = df['grid_id'].value_counts().head(50).index.tolist()
df_top = df[df['grid_id'].isin(top_cells)].copy()

# Major event flag
df_top['is_major'] = (df_top['mag'] >= 5.5).astype(int)

# Build weekly transactions: which cells had M≥5.5 events each week?
df_top_sorted = df_top.sort_values('time').set_index('time')
weekly_active = (df_top_sorted
                 .groupby([pd.Grouper(freq='7D'), 'grid_id'])['is_major']
                 .max()
                 .unstack(fill_value=0))

print(f'Weekly transaction matrix: {weekly_active.shape}')
print(f'Weeks with ≥1 active cell: {(weekly_active.sum(axis=1) > 0).sum()}')

Weekly transaction matrix: (829, 50)
Weeks with ≥1 active cell: 120


In [22]:
# if MLXTEND_AVAILABLE:
#     # ── FP-Growth Association Rules ────────────────────────────────────────
#     bool_mat = weekly_active.astype(bool)

#     MIN_SUPPORT = 0.05  # 5% of weeks
#     freq_items = fpgrowth(bool_mat, min_support=MIN_SUPPORT, use_colnames=True)

#     if len(freq_items) > 0:
#         rules = association_rules(freq_items, metric='confidence', min_threshold=0.3)
#         rules_sorted = rules.sort_values('lift', ascending=False)

#         print(f'✅ Found {len(freq_items)} frequent itemsets, {len(rules)} rules')
#         print('\nTop 10 rules (by lift):')
#         print(rules_sorted[['antecedents','consequents','support','confidence','lift']].head(10).to_string())
#     else:
#         print('No frequent itemsets found at this threshold')
#         rules_sorted = pd.DataFrame()

# else:
#     # ── Manual co-occurrence mining ────────────────────────────────────────
#     print('Using manual co-occurrence mining')

#     transactions = []
#     for _, row in weekly_active.iterrows():
#         active_cells = list(row[row > 0].index)
#         if active_cells:
#             transactions.append(active_cells)

#     # Count pairwise co-occurrences
#     pair_counts = Counter()
#     single_counts = Counter()
#     N_weeks = len(transactions)

#     for trans in transactions:
#         for c in trans:
#             single_counts[c] += 1
#         for pair in combinations(trans, 2):
#             pair_counts[tuple(sorted(pair))] += 1

#     MIN_CO = 3
#     top_pairs = [(p, c) for p, c in pair_counts.items() if c >= MIN_CO]
#     top_pairs.sort(key=lambda x: -x[1])

#     # Compute support & confidence (A → B)
#     assoc_rows = []
#     for (a, b), cnt in top_pairs[:30]:
#         support = cnt / N_weeks
#         conf_ab = cnt / single_counts[a] if single_counts[a] else 0
#         conf_ba = cnt / single_counts[b] if single_counts[b] else 0
#         lift    = (conf_ab / (single_counts[b]/N_weeks)) if single_counts[b] else 0
#         assoc_rows.append({'A': a, 'B': b, 'count': cnt,
#                            'support': round(support,4),
#                            'conf(A→B)': round(conf_ab,3),
#                            'lift': round(lift,3)})

#     rules_df = pd.DataFrame(assoc_rows).sort_values('lift', ascending=False)
#     print(f'\n✅ Top co-occurring cell pairs (min co-occurrence={MIN_CO}):')
#     print(rules_df.head(10).to_string(index=False))
#     rules_sorted = rules_df

if MLXTEND_AVAILABLE:
    # ── FP-Growth Association Rules ────────────────────────────────────────
    bool_mat = weekly_active.astype(bool)

    MIN_SUPPORT = 0.05
    freq_items = fpgrowth(bool_mat, min_support=MIN_SUPPORT, use_colnames=True)

    if len(freq_items) > 0:
        rules = association_rules(freq_items, metric='confidence', min_threshold=0.3)
        rules_sorted = rules.sort_values('lift', ascending=False)

        print(f'✅ Found {len(freq_items)} frequent itemsets, {len(rules)} rules')
        print('\nTop 10 rules (by lift):')
        print(rules_sorted[['antecedents','consequents','support','confidence','lift']].head(10).to_string())
    else:
        print('No frequent itemsets found at this threshold')
        rules_sorted = pd.DataFrame()

else:
    # ── Manual co-occurrence mining ────────────────────────────────────────
    print('Using manual co-occurrence mining')

    transactions = []
    for _, row in weekly_active.iterrows():
        active_cells = list(row[row > 0].index)
        if active_cells:
            transactions.append(active_cells)

    pair_counts = Counter()
    single_counts = Counter()
    N_weeks = len(transactions)

    for trans in transactions:
        for c in trans:
            single_counts[c] += 1
        for pair in combinations(trans, 2):
            pair_counts[tuple(sorted(pair))] += 1

    MIN_CO = 3
    top_pairs = [(p, c) for p, c in pair_counts.items() if c >= MIN_CO]
    top_pairs.sort(key=lambda x: -x[1])

    assoc_rows = []
    for (a, b), cnt in top_pairs[:30]:
        support = cnt / N_weeks if N_weeks else 0
        conf_ab = cnt / single_counts[a] if single_counts[a] else 0
        lift = (conf_ab / (single_counts[b]/N_weeks)) if single_counts[b] and N_weeks else 0

        assoc_rows.append({
            'A': a,
            'B': b,
            'count': cnt,
            'support': round(support, 4),
            'conf(A→B)': round(conf_ab, 3),
            'lift': round(lift, 3)
        })

    # FIX: handle empty case
    if assoc_rows:
        rules_df = pd.DataFrame(assoc_rows).sort_values('lift', ascending=False)
    else:
        rules_df = pd.DataFrame(columns=['A','B','count','support','conf(A→B)','lift'])

    print(f'\n✅ Top co-occurring cell pairs (min co-occurrence={MIN_CO}):')
    print(rules_df.head(10).to_string(index=False))

    rules_sorted = rules_df

Using manual co-occurrence mining

✅ Top co-occurring cell pairs (min co-occurrence=3):
Empty DataFrame
Columns: [A, B, count, support, conf(A→B), lift]
Index: []


In [23]:
# ── Association rules visualisation ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Weekly active cell heatmap (time × cells)
heatmap_data = weekly_active.tail(104).T   # last 2 years, cells as rows
if heatmap_data.shape[0] > 20:
    # Keep top-20 most active cells
    top20 = heatmap_data.sum(axis=1).nlargest(20).index
    heatmap_data = heatmap_data.loc[top20]

sns.heatmap(heatmap_data, cmap='YlOrRd', ax=axes[0],
            xticklabels=False, cbar_kws={'label': 'M≥5.5 event this week?'})
axes[0].set_title('Weekly Activity Map — Top Grid Cells (last 2 years)')
axes[0].set_xlabel('Week')
axes[0].set_ylabel('Grid Cell')

# Top rules scatter: support vs confidence, size=lift
if isinstance(rules_sorted, pd.DataFrame) and len(rules_sorted) > 0:
    if 'support' in rules_sorted.columns and 'confidence' in rules_sorted.columns:
        sc = axes[1].scatter(
            rules_sorted['support'].head(50),
            rules_sorted['confidence'].head(50),
            c=rules_sorted['lift'].head(50),
            s=100, cmap='viridis', alpha=0.8, edgecolor='white'
        )
        plt.colorbar(sc, ax=axes[1], label='Lift')
        axes[1].set_title('Association Rules — Support vs Confidence (size=lift)')
        axes[1].set_xlabel('Support')
        axes[1].set_ylabel('Confidence')
    elif 'support' in rules_sorted.columns and 'conf(A→B)' in rules_sorted.columns:
        sc = axes[1].scatter(
            rules_sorted['support'].head(30),
            rules_sorted['conf(A→B)'].head(30),
            c=rules_sorted['lift'].head(30),
            s=100, cmap='viridis', alpha=0.8, edgecolor='white'
        )
        plt.colorbar(sc, ax=axes[1], label='Lift')
        axes[1].set_title('Co-occurrence Rules — Support vs Confidence')
        axes[1].set_xlabel('Support')
        axes[1].set_ylabel('Confidence (A→B)')
else:
    axes[1].text(0.5, 0.5, 'No rules to display', ha='center', va='center')

plt.suptitle('Temporal Association Rule Mining', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('temporal_association_rules.png', dpi=150, bbox_inches='tight')
plt.show()

## 5b. Motif Detection in Time Series

A **time-series motif** is a frequently-occurring, approximately repeated subsequence. 
Motifs reveal recurring temporal patterns (e.g., aftershock sequences, seasonal bursts). 
We use a brute-force sliding-window approach with z-normalized Euclidean distance (MASS-style).


In [ ]:
# ════════════════════════════════════════════════════════════════════
# PLOT MINE-MOTIF: Example of Motif in Time Series
#
#  A motif is a short sub-sequence that appears (approx) more than once.
#  Here we find the top-2 motif pairs in the daily seismic count series.
# ════════════════════════════════════════════════════════════════════
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# ── Rebuild daily series ──
_ts_motif = (df.set_index('time').resample('D')['mag']
              .count().fillna(0).astype(float))
ts_vals = _ts_motif.values
ts_idx  = _ts_motif.index

# ── Z-normalise a sub-sequence ──
def znorm(x):
    s = x.std()
    return (x - x.mean()) / (s if s > 1e-8 else 1.0)

# ── Brute-force motif search ──
def find_top_motif_pairs(ts, m=14, exclusion_zone=None, top_k=2):
    if exclusion_zone is None:
        exclusion_zone = m // 2
    n = len(ts)
    num_sub = n - m + 1
    subs = np.array([znorm(ts[i:i+m]) for i in range(num_sub)])
    pairs   = []
    used    = set()
    dist_matrix = np.full((num_sub, num_sub), np.inf)
    for i in range(num_sub):
        for j in range(i + exclusion_zone + 1, num_sub):
            d = np.linalg.norm(subs[i] - subs[j]) / np.sqrt(m)
            dist_matrix[i, j] = d
            dist_matrix[j, i] = d
    while len(pairs) < top_k:
        dm = dist_matrix.copy()
        for u in used:
            dm[u, :] = np.inf
            dm[:, u] = np.inf
        if dm.min() == np.inf:
            break
        idx = np.unravel_index(dm.argmin(), dm.shape)
        i, j = idx
        pairs.append((i, j, dist_matrix[i, j]))
        for k in range(max(0, i - exclusion_zone),
                       min(num_sub, i + exclusion_zone + 1)):
            used.add(k)
        for k in range(max(0, j - exclusion_zone),
                       min(num_sub, j + exclusion_zone + 1)):
            used.add(k)
    return pairs

# Use last 3 years for speed
_yr_max = ts_idx.year.max()
_mask   = ts_idx.year >= _yr_max - 2
ts_sub  = ts_vals[_mask]
ts_sub_idx = ts_idx[_mask]

MOTIF_LEN = 14
print(f'Searching for motifs of length {MOTIF_LEN} in {len(ts_sub)}-day series...')
motif_pairs = find_top_motif_pairs(ts_sub, m=MOTIF_LEN, top_k=2)
print(f'Found {len(motif_pairs)} motif pair(s).')

# ── Plot ──
COLORS = [['#d4820a', '#c0392b'], ['#2176ae', '#6d28d9']]              # ← darkened all

fig = plt.figure(figsize=(20, 12))
fig.patch.set_facecolor('white')                                        
gs = gridspec.GridSpec(3, len(motif_pairs), figure=fig,
                       hspace=0.55, wspace=0.35,
                       height_ratios=[2.5, 1.5, 1.5])

# Top row: full series with motif locations highlighted
ax_full = fig.add_subplot(gs[0, :])
ax_full.set_facecolor('white')                                          
ax_full.plot(ts_sub_idx, ts_sub,
             color='#999', linewidth=0.8, alpha=0.7, label='Daily Count')  

for pair_idx, (i, j, dist) in enumerate(motif_pairs):
    c1, c2 = COLORS[pair_idx % len(COLORS)]
    for pos, col in [(i, c1), (j, c2)]:
        ax_full.axvspan(ts_sub_idx[pos],
                        ts_sub_idx[min(pos + MOTIF_LEN - 1, len(ts_sub_idx)-1)],
                        alpha=0.3, color=col,
                        label=f'Motif {pair_idx+1} (dist={dist:.2f})')

ax_full.set_title('Full Time Series with Motif Locations Highlighted',
                  color='black', fontsize=12, fontweight='bold')        
ax_full.set_ylabel('Daily Count', color='#333', fontsize=10)           
ax_full.tick_params(colors='#333')                                      
ax_full.legend(facecolor='white', edgecolor='#bbb',                    
               labelcolor='black', fontsize=8, loc='upper right')      
for sp in ax_full.spines.values(): sp.set_edgecolor('#bbb')            

# Bottom rows: each motif pair
for pair_idx, (i, j, dist) in enumerate(motif_pairs):
    c1, c2 = COLORS[pair_idx % len(COLORS)]

    # Raw subsequences
    ax_raw = fig.add_subplot(gs[1, pair_idx])
    ax_raw.set_facecolor('white')                                       
    ax_raw.plot(ts_sub[i:i+MOTIF_LEN], color=c1, linewidth=2,
                label=f'Occurrence A  @idx {i}')
    ax_raw.plot(ts_sub[j:j+MOTIF_LEN], color=c2, linewidth=2,
                linestyle='--', label=f'Occurrence B  @idx {j}')
    ax_raw.set_title(f'Motif {pair_idx+1} — Raw (dist={dist:.3f})',
                     color='black', fontsize=10, fontweight='bold')     
    ax_raw.set_xlabel('Lag (days)', color='#333', fontsize=8)          
    ax_raw.set_ylabel('Count', color='#333', fontsize=8)               
    ax_raw.tick_params(colors='#333')                                   
    ax_raw.legend(facecolor='white', edgecolor='#bbb',                 
                  labelcolor='black', fontsize=7)                       
    for sp in ax_raw.spines.values(): sp.set_edgecolor('#bbb')         

    # Z-normalised
    ax_znorm = fig.add_subplot(gs[2, pair_idx])
    ax_znorm.set_facecolor('white')                                     
    ax_znorm.plot(znorm(ts_sub[i:i+MOTIF_LEN]), color=c1, linewidth=2)
    ax_znorm.plot(znorm(ts_sub[j:j+MOTIF_LEN]), color=c2, linewidth=2, linestyle='--')
    ax_znorm.fill_between(range(MOTIF_LEN),
                          znorm(ts_sub[i:i+MOTIF_LEN]),
                          znorm(ts_sub[j:j+MOTIF_LEN]),
                          alpha=0.15, color='#555')                     (was #ffffff, invisible on white)
    ax_znorm.set_title(f'Motif {pair_idx+1} — Z-Normalised (shape comparison)',
                       color='black', fontsize=10, fontweight='bold')   
    ax_znorm.set_xlabel('Lag (days)', color='#333', fontsize=8)        
    ax_znorm.set_ylabel('Z-score', color='#333', fontsize=8)           
    ax_znorm.tick_params(colors='#333')                                 
    for sp in ax_znorm.spines.values(): sp.set_edgecolor('#bbb')       

fig.suptitle(f'🔄  Motif Detection in Time Series (window = {MOTIF_LEN} days)\n'
             'Top recurrent patterns in daily seismic event counts',
             color='black', fontsize=13, fontweight='bold', y=1.01)    

plt.savefig('plot_MINE_motif.png', dpi=150,
            bbox_inches='tight', facecolor='white')                     
plt.show()
print('✅  Motif plot saved.')


Searching for motifs of length 14 in 877-day series...
Found 2 motif pair(s).
✅  Motif plot saved.


## 6. Comprehensive Time-Series Dashboard

In [27]:
# ── Comprehensive temporal dashboard ─────────────────────────────────────
fig = plt.figure(figsize=(20, 16))
gs  = gridspec.GridSpec(4, 3, figure=fig, hspace=0.5, wspace=0.35)

# 1. Annual event counts
ax1 = fig.add_subplot(gs[0, :])
annual = df.set_index('time').resample('YE')['mag'].count()
ax1.bar(annual.index.year, annual.values, color='steelblue', alpha=0.8, edgecolor='white')
ax1.set_title('Annual Earthquake Count (Global)')
ax1.set_xlabel('Year')
ax1.set_ylabel('Count')

# 2. Monthly event counts (recent 5 years)
ax2 = fig.add_subplot(gs[1, :])
recent_start = df['time'].max() - pd.Timedelta(days=365*5)
recent = df[df['time'] >= recent_start].set_index('time').resample('ME')['mag'].count()
ax2.fill_between(recent.index, recent.values, alpha=0.5, color='teal')
ax2.plot(recent.index, recent.values, color='teal', linewidth=1.2)
# Mark PELT change points in this range
for d in cp_dates_count:
    if d >= recent_start:
        ax2.axvline(d, color='red', linestyle='--', alpha=0.7, linewidth=1.2, label='PELT CP')
ax2.set_title('Monthly Event Count — Last 5 Years (with PELT change points)')
ax2.set_ylabel('Events/month')

# 3. Magnitude distribution over time (scatter)
ax3 = fig.add_subplot(gs[2, :2])
sc = ax3.scatter(df['time'], df['mag'],
                  c=df['depth'].clip(0, 700), cmap='plasma_r',
                  s=1.5, alpha=0.3, vmin=0, vmax=700)
plt.colorbar(sc, ax=ax3, label='Depth (km)')
ax3.set_title('Magnitude Timeline (coloured by depth)')
ax3.set_ylabel('Magnitude')
ax3.axhline(6.0, color='red', linestyle='--', linewidth=0.8, alpha=0.7)

# 4. b-value over time (if available)
ax4 = fig.add_subplot(gs[2, 2])
if 'b_value_rolling' in df.columns:
    bv_month = df.set_index('time').resample('ME')['b_value_rolling'].mean().dropna()
    ax4.plot(bv_month.index, bv_month.values, color='darkgreen', linewidth=1.0)
    ax4.fill_between(bv_month.index, bv_month.values, alpha=0.2, color='green')
    ax4.axhline(1.0, color='gray', linestyle='--', linewidth=0.8)
ax4.set_title('Rolling b-value (90-day)')
ax4.set_ylabel('b-value')

# 5. Hour-of-day distribution
ax5 = fig.add_subplot(gs[3, 0])
df['hour'] = df['time'].dt.hour
hour_counts = df.groupby('hour').size()
ax5.bar(hour_counts.index, hour_counts.values, color='mediumpurple', edgecolor='white', alpha=0.8)
ax5.set_title('Earthquakes by Hour of Day (UTC)')
ax5.set_xlabel('Hour')
ax5.set_ylabel('Count')

# 6. Day-of-week distribution
ax6 = fig.add_subplot(gs[3, 1])
df['dow'] = df['time'].dt.dayofweek
dow_counts = df.groupby('dow').size()
days = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
ax6.bar(days, [dow_counts.get(i, 0) for i in range(7)],
        color='coral', edgecolor='white', alpha=0.8)
ax6.set_title('Earthquakes by Day of Week')
ax6.set_ylabel('Count')

# 7. IET distribution vs Poisson
ax7 = fig.add_subplot(gs[3, 2])
iet_h = df['inter_event_time_hrs'].clip(upper=df['inter_event_time_hrs'].quantile(0.99))
hist_vals, bin_edges = np.histogram(iet_h, bins=40, density=True)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
ax7.bar(bin_centers, hist_vals, width=np.diff(bin_edges), color='steelblue', alpha=0.7, label='Observed IET')
# Fit exponential (expected for Poisson process)
lam = 1 / (iet_h.mean() + 1e-9)
exp_pdf = lam * np.exp(-lam * bin_centers)
ax7.plot(bin_centers, exp_pdf, 'r-', linewidth=2, label='Exponential (Poisson)')
ax7.set_title('IET vs Poisson Process')
ax7.set_xlabel('IET (hours)')
ax7.set_ylabel('Density')
ax7.legend(fontsize=9)

plt.suptitle('Temporal Data Mining — Comprehensive Dashboard\nTeam Member 2 | AID843', 
             fontsize=16, fontweight='bold', y=1.01)
plt.savefig('temporal_mining_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Temporal mining dashboard saved')

✅ Temporal mining dashboard saved


## 7. Summary of Mining Results

In [24]:
print('=' * 65)
print('  TEMPORAL DATA MINING — SUMMARY')
print('=' * 65)

print(f'''
1. Sequence Mining (PrefixSpan)
   Sequences      : {len(sequences):,}
   Avg seq length : {np.mean([len(s) for s in sequences]):.1f}
   Frequent pats  : {len(patterns_sorted)}
   Most common    : {" → ".join(patterns_sorted[0][1])} (support: {patterns_sorted[0][0]})

2. Periodicity Mining (FFT)
   Top period     : {top_periods[0]:.1f} days ({top_periods[0]/30.44:.1f} months)
   2nd period     : {top_periods[1]:.1f} days
   3rd period     : {top_periods[2]:.1f} days
   Annual seasonal trend detected via decomposition ✓

3. Change Point Detection (PELT + BOCPD)
   PELT (count)   : {len(bkps_count)-1} change points
   PELT (mag)     : {len(bkps_mag)-1} change points
   BOCPD CPs      : {len(bocpd_dates)} above threshold={bocpd_threshold}

4. Temporal Association Rules
   Grid cells     : {len(top_cells)} top cells analysed
   Weekly windows : {len(weekly_active)} weeks
   Rules output saved to temporal_association_rules.png
''')
print('=' * 65)
print('\nOutput files:')
output_files = [
    'sequence_mining.png',
    'periodicity_analysis.png',
    'change_point_detection.png',
    'temporal_association_rules.png',
    'temporal_mining_dashboard.png'
]
for f in output_files:
    print(f'  ✅ {f}')

  TEMPORAL DATA MINING — SUMMARY

1. Sequence Mining (PrefixSpan)
   Sequences      : 176
   Avg seq length : 5.6
   Frequent pats  : 28
   Most common    : L → L (support: 165)

2. Periodicity Mining (FFT)
   Top period     : 3092.7 days (101.6 months)
   2nd period     : 1546.3 days
   3rd period     : 386.6 days
   Annual seasonal trend detected via decomposition ✓

3. Change Point Detection (PELT + BOCPD)
   PELT (count)   : 4 change points
   PELT (mag)     : 3 change points
   BOCPD CPs      : 0 above threshold=0.3

4. Temporal Association Rules
   Grid cells     : 50 top cells analysed
   Weekly windows : 829 weeks
   Rules output saved to temporal_association_rules.png


Output files:
  ✅ sequence_mining.png
  ✅ periodicity_analysis.png
  ✅ change_point_detection.png
  ✅ temporal_association_rules.png
  ✅ temporal_mining_dashboard.png
